<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎬 LTX-2.3 22B Distilled (quanto int8)</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>Text & Image to Video with Audio | Wan2GP Engine + INT8 Tensor Core Kernels | Dual T4</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### What is LTX-2.3 22B Distilled?

**LTX-2.3 22B Distilled** is a state-of-the-art open video + audio generation model, running on free Kaggle GPU T4 x2 via the **Wan2GP** engine with INT8 tensor-core kernels and both GPUs in use.

| Feature | Details |
|----|----|
| Model | LTX-2.3 22B Distilled (quanto int8) |
| Engine | Wan2GP + mmgp Profile 4 + Comfy Kitchen INT8 kernels |
| Pipeline | Two-stage: 8 steps (half-res) → 2x spatial upscale → 3 steps (refine) → FP32 VAE |
| GPU 0 | 22B transformer (10.8 GB resident, rest streamed) + FP32 VAE decoder |
| GPU 1 | Gemma 3 12B text encoder + spatial upsampler + video encoder |
| Modes | Text-to-Video, Image-to-Video, First & Last Frame, synced audio |
| Output | Saved to `/kaggle/working/outputs` (Kaggle Output panel) |

### Quick Start
1. **Settings → Accelerator → GPU T4 x2**
2. **Turn on Internet** in Settings sidebar
3. Run all cells in order
4. Open the **Gradio** public link, or the **Cloudflare** tunnel link if Gradio's link does not load
5. Use detailed prompts (subject, action, setting, camera, lighting) for the best results

In [ ]:
# Step 1: Environment Setup, Dependencies, Model Downloads & Patches
import os
import sys
import gc
import psutil
import shutil
import subprocess
import warnings

# 1. Configure High-Performance Environment & Silence Warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TQDM_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("=" * 65)
print("🎬 LTX-2.3 22B Distilled - Video Generator")
print("📺 Created by: AIQUEST Academy")
print("🔗 YouTube: @AIQuestAcademy | X: @AIQuestAcademy")
print("=" * 65)
print("=== Kaggle GPU T4 x2 Environment Setup ===")
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")

# 2. Hardware and GPU Verification
try:
    smi = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
    print(f"✅ GPU Detected: {smi}")
except Exception:
    print("WARNING: No GPU. Go to Settings → Accelerator → GPU T4 x2")

# Drop filesystem caches to free RAM
os.system("echo 3 | sudo -n tee /proc/sys/vm/drop_caches > /dev/null 2>&1")
os.system("echo 1 | sudo -n tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1")
gc.collect()

# 3. Clone Wan2GP Repository
REPO_DIR = "Wan2GP"
if not os.path.exists(REPO_DIR):
    print("\n📥 Cloning Wan2GP repository...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DeepBeepMeep/Wan2GP.git", REPO_DIR], check=True)
    print("✅ Wan2GP cloned successfully.")
else:
    print("\n📂 Wan2GP repository already present.")

# 4. Install Dependencies
print("📦 Installing optimized Python dependencies...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--timeout", "120", "--retries", "5",
    "-q", "--no-warn-conflicts", "-r", f"{REPO_DIR}/requirements.txt"
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--timeout", "120", "--retries", "5",
    "-q", "--no-warn-conflicts", "mmgp", "gradio>=5.0.0", "soundfile", "hf_transfer"
], check=True)
print("✅ Python dependencies installed.")

# 5. Patch Wan2GP Runtime Bugs (NoneType input_video_strength & source dtype)
ltx2_path = os.path.join(REPO_DIR, "models/ltx2/ltx2.py")
if os.path.exists(ltx2_path):
    with open(ltx2_path, "r") as f:
        content = f.read()

    # Patch 1: NoneType in input_video_strength
    old_line1 = "input_video_strength = max(0.0, min(1.0, input_video_strength))"
    new_line1 = "input_video_strength = max(0.0, min(1.0, input_video_strength)) if input_video_strength is not None else 1.0"
    if old_line1 in content:
        content = content.replace(old_line1, new_line1)

    # Patch 2: Use self.dtype for upsampler source instead of hardcoded bfloat16
    old_line2 = "source = source.to(device=self.device, dtype=torch.bfloat16)"
    new_line2 = "source = source.to(device=self.device, dtype=self.dtype)"
    if old_line2 in content:
        content = content.replace(old_line2, new_line2)

    with open(ltx2_path, "w") as f:
        f.write(content)
    print("✅ Patched Wan2GP ltx2.py runtime routines successfully.")

# Patch 3: Wan2GP Spatial Upsampler Dtype Auto-Alignment (Prevents BFloat16/Half conv3d mismatch)
upsampler_file = os.path.join(REPO_DIR, "models/ltx2/ltx_core/model/upsampler/model.py")
if os.path.exists(upsampler_file):
    with open(upsampler_file, "r") as f:
        ucontent = f.read()
    old_up = "latent = upsampler(latent)"
    new_up = ("upsampler_dtype = next(upsampler.parameters()).dtype if any(True for _ in upsampler.parameters()) else torch.float16\n"
              "    if latent.dtype != upsampler_dtype:\n"
              "        latent = latent.to(upsampler_dtype)\n"
              "    latent = upsampler(latent)")
    if old_up in ucontent:
        ucontent = ucontent.replace(old_up, new_up)
    with open(upsampler_file, "w") as f:
        f.write(ucontent)
    print("✅ Patched Wan2GP spatial upsampler dtype auto-alignment successfully.")

# Patch 4: Wan2GP VideoDecoder & VideoEncoder Dtype Auto-Alignment (Prevents VAE decode/encode conv mismatch)
vae_file = os.path.join(REPO_DIR, "models/ltx2/ltx_core/model/video_vae/video_vae.py")
if os.path.exists(vae_file):
    with open(vae_file, "r") as f:
        vcontent = f.read()
    old_v = "sample = self.conv_in(sample, causal=self.causal)"
    new_v = ("conv_param = next(self.conv_in.parameters(), None)\n"
             "        if conv_param is not None and sample.dtype != conv_param.dtype:\n"
             "            sample = sample.to(conv_param.dtype)\n"
             "        sample = self.conv_in(sample, causal=self.causal)")
    if old_v in vcontent:
        vcontent = vcontent.replace(old_v, new_v)
    old_enc = "sample = self.conv_in(sample)"
    new_enc = ("enc_param = next(self.conv_in.parameters(), None)\n"
               "        if enc_param is not None and sample.dtype != enc_param.dtype:\n"
               "            sample = sample.to(enc_param.dtype)\n"
               "        sample = self.conv_in(sample)")
    if old_enc in vcontent:
        vcontent = vcontent.replace(old_enc, new_enc)
    with open(vae_file, "w") as f:
        f.write(vcontent)
    print("✅ Patched Wan2GP video decoder & encoder dtype auto-alignment successfully.")

# Patch 5: Wan2GP AudioDecoder & Vocoder Dtype Auto-Alignment (Prevents BFloat16/Half audio mismatch)
audio_vae_file = os.path.join(REPO_DIR, "models/ltx2/ltx_core/model/audio_vae/audio_vae.py")
if os.path.exists(audio_vae_file):
    with open(audio_vae_file, "r") as f:
        acontent = f.read()
    old_a = "decoded_audio = audio_decoder(latent)"
    new_a = ("decoder_param = next(audio_decoder.parameters(), None)\n"
             "    if decoder_param is not None and latent.dtype != decoder_param.dtype:\n"
             "        latent = latent.to(decoder_param.dtype)\n"
             "    decoded_audio = audio_decoder(latent)\n"
             "    vocoder_param = next(vocoder.parameters(), None)\n"
             "    if vocoder_param is not None and decoded_audio.dtype != vocoder_param.dtype:\n"
             "        decoded_audio = decoded_audio.to(vocoder_param.dtype)")
    if old_a in acontent:
        acontent = acontent.replace(old_a, new_a)
    with open(audio_vae_file, "w") as f:
        f.write(acontent)
    print("✅ Patched Wan2GP audio decoder & vocoder dtype auto-alignment successfully.")

# Patch 6: Wan2GP Vocoder STFT & Filter Dtype Auto-Alignment (Prevents BFloat16 buffer mismatch with Half input)
vocoder_file = os.path.join(REPO_DIR, "models/ltx2/ltx_core/model/audio_vae/vocoder.py")
if os.path.exists(vocoder_file):
    with open(vocoder_file, "r") as f:
        voc_content = f.read()
    old_stft = "spec = F.conv1d(y, self.forward_basis, stride=self.hop_length, padding=0)"
    new_stft = ("forward_basis = self.forward_basis.to(dtype=y.dtype, device=y.device) if self.forward_basis.dtype != y.dtype else self.forward_basis\n"
                "        spec = F.conv1d(y, forward_basis, stride=self.hop_length, padding=0)")
    if old_stft in voc_content:
        voc_content = voc_content.replace(old_stft, new_stft)
    old_low = "return F.conv1d(x, self.filter.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)"
    new_low = ("filt = self.filter.to(dtype=x.dtype, device=x.device) if self.filter.dtype != x.dtype else self.filter\n"
               "        return F.conv1d(x, filt.expand(n_channels, -1, -1), stride=self.stride, groups=n_channels)")
    if old_low in voc_content:
        voc_content = voc_content.replace(old_low, new_low)
    with open(vocoder_file, "w") as f:
        f.write(voc_content)
    print("✅ Patched Wan2GP vocoder STFT & filter buffer auto-alignment successfully.")

# 6. High-Speed Model Checkpoint Downloads (Disk-aware with /kaggle/tmp symlinks)
from huggingface_hub import hf_hub_download

REPO = "DeepBeepMeep/LTX-2"
MODEL_DIR = os.path.join(REPO_DIR, "models")
TMP_DIR = "/kaggle/tmp/models" if os.path.exists("/kaggle") else os.path.abspath("tmp_models")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# Required large files (download to /kaggle/tmp, symlink to Wan2GP/models)
# Note: ltx-2.3-22b-distilled-lora-384.safetensors (7.6 GB) is excluded as weights are baked into quanto int8
LARGE_FILES = [
    ("ltx-2.3-22b-distilled_diffusion_model_quanto_int8.safetensors", "19.4 GB"),
    ("ltx-2.3-22b_embeddings_connector.safetensors", "4.0 GB"),
    ("ltx-2.3-22b_text_embedding_projection.safetensors", "2.3 GB"),
    ("ltx-2.3-22b_vae.safetensors", "1.5 GB"),
]

print("\n🚀 Downloading large model checkpoints...")
for fname, fsize in LARGE_FILES:
    dest = os.path.join(MODEL_DIR, fname)
    if os.path.exists(dest):
        print(f"  ✓ Found: {fname}")
        continue
    print(f"  ⚡ Downloading {fname} ({fsize})...")
    hf_hub_download(repo_id=REPO, filename=fname, local_dir=TMP_DIR)
    actual = os.path.join(TMP_DIR, fname)
    if not os.path.exists(dest):
        os.symlink(actual, dest)
    print(f"  ✅ Linked: {fname}")

# Required small files
SMALL_FILES = [
    "ltx-2.3-22b_audio_vae.safetensors",
    "ltx-2.3-22b_vocoder.safetensors",
    "ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
]

print("\n🚀 Downloading auxiliary pipeline models...")
for fname in SMALL_FILES:
    dest = os.path.join(MODEL_DIR, fname)
    if os.path.exists(dest):
        print(f"  ✓ Found: {fname}")
        continue
    print(f"  ⚡ Downloading {fname}...")
    hf_hub_download(repo_id=REPO, filename=fname, local_dir=MODEL_DIR)
    print(f"  ✅ Saved: {fname}")

# Gemma text encoder folder
GEMMA_FOLDER = "gemma-3-12b-it-qat-q4_0-unquantized"
GEMMA_FILES = [
    "gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors",
    "added_tokens.json",
    "chat_template.json",
    "config_light.json",
    "generation_config.json",
    "preprocessor_config.json",
    "processor_config.json",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer.model",
    "tokenizer_config.json",
]

gemma_dest = os.path.join(MODEL_DIR, GEMMA_FOLDER)
gemma_tmp = os.path.join(TMP_DIR, GEMMA_FOLDER)

print("\n🚀 Downloading Gemma text encoder weights...")
if os.path.exists(gemma_dest):
    print(f"  ✓ Found: {GEMMA_FOLDER}/")
else:
    os.makedirs(gemma_tmp, exist_ok=True)
    for gf in GEMMA_FILES:
        tmp_file = os.path.join(gemma_tmp, gf)
        if os.path.exists(tmp_file):
            continue
        hf_hub_download(repo_id=REPO, filename=f"{GEMMA_FOLDER}/{gf}", local_dir=TMP_DIR)
    if not os.path.exists(gemma_dest):
        os.symlink(gemma_tmp, gemma_dest)
    print(f"  ✅ Linked: {GEMMA_FOLDER}/")

# Clean Hugging Face download cache to conserve Kaggle disk
for cdir in [os.path.join(MODEL_DIR, ".cache"), os.path.join(TMP_DIR, ".cache")]:
    if os.path.exists(cdir):
        shutil.rmtree(cdir, ignore_errors=True)

print("\n" + "=" * 65)
print("✅ All checkpoints verified and ready for generation!")
print("=" * 65)

In [ ]:
# Step 2: High-Performance Engine & Branded Gradio UI
# Writes run_ltx.py and launches the Gradio server with streaming real-time progress
import os
import sys
import gc
import subprocess

RUN_SCRIPT_CODE = '''import gc
import os
import sys
import time
import json
import random
import tempfile
import glob
import traceback
import subprocess
import numpy as np
import psutil
from PIL import Image
import warnings

# Suppress log clutter and set memory optimizations
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TQDM_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"

# ---- Bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
# Videos are saved in the working folder (/kaggle/working/outputs), visible in Kaggle's Output panel
# even if the Gradio UI loses its connection during a long generation.
OUTPUT_DIR = os.path.join(os.path.dirname(WAN2GP_DIR), "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ.setdefault("GRADIO_TEMP_DIR", os.path.join(OUTPUT_DIR, ".gradio_cache"))
if WAN2GP_DIR not in sys.path:
    sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

import torch
import gradio as gr
from shared.utils.audio_video import save_video

# ==== GPU & Hardware Verification ====
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
ram = psutil.virtual_memory()
print("=" * 65)
print("🎬 LTX-2.3 22B Distilled - Video Generation Engine")
print(f"⚡ GPU: {gpu_name} ({total_vram_gb:.1f} GB VRAM) | RAM: {ram.total / 1024**3:.1f} GB ({ram.available / 1024**3:.1f} GB free)")
print("=" * 65)
sys.stdout.flush()

# ==== Force Fast SDPA Backend for NVIDIA T4 (Turing FP16 + Cutlass) ====
torch.backends.cuda.enable_flash_sdp(False)        # T4 SM75 does not support FlashAttention-2
torch.backends.cuda.enable_mem_efficient_sdp(True)  # Fast Cutlass attention
torch.backends.cuda.enable_math_sdp(True)           # Enable math attention for small prompt sequences
torch.backends.cudnn.benchmark = False              # Disable dynamic conv benchmarking to prevent step 1 stall
if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
    torch.backends.cuda.matmul.allow_tf32 = True

# ==== Patch SDPA in PyTorch & Transformers to handle mixed dtypes in Gemma 3 ====
_orig_scaled_dot_product_attention = torch.nn.functional.scaled_dot_product_attention

def _safe_scaled_dot_product_attention(query, key, value, *args, **kwargs):
    target_dtype = query.dtype
    if key.dtype != target_dtype:
        key = key.to(target_dtype)
    if value.dtype != target_dtype:
        value = value.to(target_dtype)
    attn_mask = kwargs.get("attn_mask", None)
    if attn_mask is not None and torch.is_tensor(attn_mask) and attn_mask.is_floating_point() and attn_mask.dtype != target_dtype:
        kwargs["attn_mask"] = attn_mask.to(target_dtype)
    return _orig_scaled_dot_product_attention(query, key, value, *args, **kwargs)

torch.nn.functional.scaled_dot_product_attention = _safe_scaled_dot_product_attention

try:
    import transformers.integrations.sdpa_attention as _sdpa_mod
    _orig_sdpa_forward = _sdpa_mod.sdpa_attention_forward

    def _safe_sdpa_forward(module, query, key, value, *args, **kwargs):
        target_dtype = query.dtype
        if key.dtype != target_dtype:
            key = key.to(target_dtype)
        if value.dtype != target_dtype:
            value = value.to(target_dtype)
        return _orig_sdpa_forward(module, query, key, value, *args, **kwargs)

    _sdpa_mod.sdpa_attention_forward = _safe_sdpa_forward
except Exception:
    pass

# ==== Patch Upsampler, VAE, VideoEncoder, AudioDecoder & Vocoder for FP16 compatibility ====
try:
    import models.ltx2.ltx_core.model.upsampler.model as _upsampler_mod
    _orig_upsample_video = _upsampler_mod.upsample_video

    def _safe_upsample_video(latent, video_encoder, upsampler):
        latent = video_encoder.per_channel_statistics.un_normalize(latent)
        upsampler_param = next(upsampler.parameters(), None)
        target_dtype = upsampler_param.dtype if upsampler_param is not None else torch.float16
        if latent.dtype != target_dtype:
            latent = latent.to(target_dtype)
        latent = upsampler(latent)
        latent = video_encoder.per_channel_statistics.normalize(latent)
        return latent

    _upsampler_mod.upsample_video = _safe_upsample_video

    _orig_upsampler_forward = _upsampler_mod.LatentUpsampler.forward

    def _safe_upsampler_forward(self, latent, *args, **kwargs):
        target_param = getattr(self, "initial_conv", None)
        if target_param is not None and hasattr(target_param, "weight") and latent.dtype != target_param.weight.dtype:
            latent = latent.to(target_param.weight.dtype)
        return _orig_upsampler_forward(self, latent, *args, **kwargs)

    _upsampler_mod.LatentUpsampler.forward = _safe_upsampler_forward
except Exception as e:
    print(f"⚠️ Upsampler monkeypatch note: {e}")

try:
    import models.ltx2.ltx_core.model.video_vae.video_vae as _video_vae_mod
    _orig_video_decoder_forward = _video_vae_mod.VideoDecoder.forward

    def _safe_video_decoder_forward(self, sample, *args, **kwargs):
        conv_param = next(self.parameters(), None)
        if conv_param is not None and sample.dtype != conv_param.dtype:
            sample = sample.to(conv_param.dtype)
        return _orig_video_decoder_forward(self, sample, *args, **kwargs)

    _video_vae_mod.VideoDecoder.forward = _safe_video_decoder_forward

    _orig_video_encoder_forward = _video_vae_mod.VideoEncoder.forward

    def _safe_video_encoder_forward(self, sample, *args, **kwargs):
        # Encode in the encoder's own dtype (FP32), then hand back the caller's dtype so image
        # conditioning latents (I2V start / end frames) match the BF16 diffusion state.
        # The encoder may also live on the second GPU: run it there and return to the caller's device.
        in_dtype, in_device = sample.dtype, sample.device
        enc_param = next(self.parameters(), None)
        if enc_param is not None and enc_param.device.type == "cuda" and sample.device != enc_param.device:
            sample = sample.to(enc_param.device)
        if enc_param is not None and sample.dtype != enc_param.dtype:
            sample = sample.to(enc_param.dtype)
        with torch.cuda.device(sample.device if sample.is_cuda else torch.cuda.current_device()):
            out = _orig_video_encoder_forward(self, sample, *args, **kwargs)
        if torch.is_tensor(out) and out.is_floating_point():
            out = out.to(device=in_device, dtype=in_dtype if in_dtype.is_floating_point else out.dtype)
        return out

    _video_vae_mod.VideoEncoder.forward = _safe_video_encoder_forward
except Exception as e:
    print(f"⚠️ Video VAE monkeypatch note: {e}")

def _safe_decode_audio(latent, audio_decoder, vocoder):
    try:
        if latent is not None and torch.is_tensor(latent) and latent.dtype != torch.float16:
            latent = latent.to(torch.float16)

        if audio_decoder is not None and isinstance(audio_decoder, torch.nn.Module):
            for b_name, b_val in audio_decoder.named_buffers():
                if b_val is not None and torch.is_tensor(b_val) and b_val.is_floating_point() and b_val.dtype != torch.float16:
                    b_val.data = b_val.data.to(torch.float16)

        decoded_audio = audio_decoder(latent)

        if decoded_audio is not None and torch.is_tensor(decoded_audio) and decoded_audio.dtype != torch.float16:
            decoded_audio = decoded_audio.to(torch.float16)

        if vocoder is not None and isinstance(vocoder, torch.nn.Module):
            for b_name, b_val in vocoder.named_buffers():
                if b_val is not None and torch.is_tensor(b_val) and b_val.is_floating_point() and b_val.dtype != torch.float16:
                    b_val.data = b_val.data.to(torch.float16)

        decoded_audio = vocoder(decoded_audio).squeeze(0).float()
        return decoded_audio
    except Exception as e:
        print(f"  ⚠️ Audio decode error caught: {e}")
        return None

try:
    import models.ltx2.ltx_core.model.audio_vae.audio_vae as _audio_vae_mod
    import models.ltx2.ltx_core.model.audio_vae as _audio_vae_pkg

    if hasattr(_audio_vae_mod, "decode_audio"):
        _audio_vae_mod.decode_audio = _safe_decode_audio
    if hasattr(_audio_vae_pkg, "decode_audio"):
        _audio_vae_pkg.decode_audio = _safe_decode_audio

    _orig_audio_decoder_forward = _audio_vae_mod.AudioDecoder.forward

    def _safe_audio_decoder_forward(self, sample, *args, **kwargs):
        if sample is not None and torch.is_tensor(sample) and sample.dtype != torch.float16:
            sample = sample.to(torch.float16)
        for b_name, b_val in self.named_buffers():
            if b_val is not None and torch.is_tensor(b_val) and b_val.is_floating_point() and b_val.dtype != torch.float16:
                b_val.data = b_val.data.to(torch.float16)
        return _orig_audio_decoder_forward(self, sample, *args, **kwargs)

    _audio_vae_mod.AudioDecoder.forward = _safe_audio_decoder_forward

    import models.ltx2.ltx_core.model.audio_vae.vocoder as _vocoder_mod
    _orig_vocoder_forward = _vocoder_mod.Vocoder.forward

    def _safe_vocoder_forward(self, x, *args, **kwargs):
        if x is not None and torch.is_tensor(x) and x.dtype != torch.float16:
            x = x.to(torch.float16)
        for b_name, b_val in self.named_buffers():
            if b_val is not None and torch.is_tensor(b_val) and b_val.is_floating_point() and b_val.dtype != torch.float16:
                b_val.data = b_val.data.to(torch.float16)
        return _orig_vocoder_forward(self, x, *args, **kwargs)

    _vocoder_mod.Vocoder.forward = _safe_vocoder_forward

    if hasattr(_vocoder_mod, "VocoderWithBWE"):
        _orig_vocoder_bwe_forward = _vocoder_mod.VocoderWithBWE.forward
        def _safe_vocoder_bwe_forward(self, x, *args, **kwargs):
            if x is not None and torch.is_tensor(x) and x.dtype != torch.float16:
                x = x.to(torch.float16)
            for b_name, b_val in self.named_buffers():
                if b_val is not None and torch.is_tensor(b_val) and b_val.is_floating_point() and b_val.dtype != torch.float16:
                    b_val.data = b_val.data.to(torch.float16)
            return _orig_vocoder_bwe_forward(self, x, *args, **kwargs)
        _vocoder_mod.VocoderWithBWE.forward = _safe_vocoder_bwe_forward

    if hasattr(_vocoder_mod, "_STFTFn"):
        _orig_stft_forward = _vocoder_mod._STFTFn.forward
        def _safe_stft_forward(self, y, *args, **kwargs):
            if hasattr(self, "forward_basis") and torch.is_tensor(self.forward_basis) and self.forward_basis.dtype != y.dtype:
                self.forward_basis.data = self.forward_basis.data.to(dtype=y.dtype, device=y.device)
            return _orig_stft_forward(self, y, *args, **kwargs)
        _vocoder_mod._STFTFn.forward = _safe_stft_forward

    if hasattr(_vocoder_mod, "LowPassFilter1d"):
        _orig_lowpass_forward = _vocoder_mod.LowPassFilter1d.forward
        def _safe_lowpass_forward(self, x, *args, **kwargs):
            if hasattr(self, "filter") and torch.is_tensor(self.filter) and self.filter.dtype != x.dtype:
                self.filter.data = self.filter.data.to(dtype=x.dtype, device=x.device)
            return _orig_lowpass_forward(self, x, *args, **kwargs)
        _vocoder_mod.LowPassFilter1d.forward = _safe_lowpass_forward
except Exception as e:
    print(f"⚠️ Audio VAE monkeypatch note: {e}")

# ==== Load Model via Wan2GP ====
print("\\n📦 Loading LTX-2.3 22B Distilled pipeline (INT8 Tensor Cores)...")
sys.stdout.flush()

from mmgp import offload
from shared.utils import files_locator as fl

# ==== INT8 Tensor Core kernels (Wan2GP's own "INT8 Kernels: Auto" setting) ====
# Without this, every quanto int8 layer dequantizes to BF16 and runs a BF16 matmul, which the
# T4 can only do on its slow CUDA cores. "auto" tries Comfy Kitchen (INT8 GEMM, sm_75+), then
# Triton, and falls back to plain PyTorch if neither passes its self-test.
int8_backend = None
try:
    from shared.kernels import int8_backend
    int8_backend.configure("auto", 1)
except Exception as e:
    print(f"⚠️ INT8 kernels unavailable, using PyTorch matmul: {e}")

fl.set_checkpoints_paths(["models", "ckpts", "."])

from models.ltx2.ltx2_handler import family_handler

base_model_type = "ltx2_22B"
model_def = {"ltx2_pipeline": "distilled"}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

gemma_folder = "models/gemma-3-12b-it-qat-q4_0-unquantized"
gemma_files = sorted(glob.glob(os.path.join(gemma_folder, "*.safetensors")))
quanto_files = [f for f in gemma_files if "quanto" in f]
text_encoder_file = quanto_files[0] if quanto_files else (gemma_files[0] if gemma_files else None)
if not text_encoder_file:
    raise FileNotFoundError(f"Missing text encoder in {gemma_folder}.")

transformer_path = os.path.join("models", "ltx-2.3-22b-distilled_diffusion_model_quanto_int8.safetensors")
if not os.path.isfile(transformer_path):
    raise FileNotFoundError(f"Transformer weights not found at {transformer_path}.")

ltx2_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type="ltx2_22B_distilled",
    base_model_type=base_model_type,
    model_def=model_def,
    dtype=torch.float16,         # FP16 activates T4 Tensor Cores
    VAE_dtype=torch.float32,     # Float32 VAE prevents NaNs and black frames
    text_encoder_filename=text_encoder_file,
)

# Convert floating-point buffers in the audio modules to float16 (the audio patches above expect fp16).
# The video encoder / upsampler are NOT touched here: they run in FP32 (see below).
for model_key in ["audio_decoder", "vocoder", "audio_encoder"]:
    mod = pipe.get(model_key)
    if mod is not None and isinstance(mod, torch.nn.Module):
        for b_name, b_val in mod.named_buffers():
            if b_val is not None and torch.is_tensor(b_val) and b_val.is_floating_point() and b_val.dtype != torch.float16:
                b_val.data = b_val.data.to(torch.float16)

try:
    import models.ltx2.ltx_pipelines.distilled as _distilled_mod
    _distilled_mod.vae_decode_audio = _safe_decode_audio
except Exception:
    pass

has_upscaler = pipe.get("spatial_upsampler") is not None
print(f"✅ Pipeline components active | Spatial 2x Upscaler: {'Loaded' if has_upscaler else 'Missing'}")
sys.stdout.flush()

# Run the video VAE (decoder + encoder) and the latent upsampler in true FP32.
# Their checkpoints are stored in BF16 and mmgp only ever downcasts FP32 weights, so setting
# _lock_dtype alone left them in BF16, a dtype the T4 (sm_75) has no native kernels for.
# Convert first, then lock + tag so mmgp keeps them (and their inputs) in FP32.
for _key in ("video_decoder", "video_encoder", "spatial_upsampler"):
    _mod = getattr(ltx2_model, _key, None)
    if _mod is None:
        continue
    _mod.to(torch.float32)
    _mod._model_dtype = torch.float32
    for m in _mod.modules():
        m._lock_dtype = torch.float32
_vae_dtypes = {p.dtype for p in ltx2_model.video_decoder.parameters()}
print(f"🔒 Video VAE + Spatial Upsampler converted to FP32 (decoder weight dtypes: {_vae_dtypes}).")
sys.stdout.flush()

# ==== Stage probes: log latent / pixel statistics so a bad stage is visible in one run ====
# distilled.py binds upsample_video / vae_decode_video_to_tensor by name at import,
# so we wrap the names inside that module (patching the source modules has no effect).
def _latent_stats(tag, t):
    try:
        t = t[0] if isinstance(t, (list, tuple)) else t
        f = t.detach().float()
        nonfinite = int((~torch.isfinite(f)).sum().item())
        f = torch.nan_to_num(f)
        print(f"  🔬 {tag}: shape={tuple(t.shape)} dtype={t.dtype} mean={f.mean().item():+.3f} std={f.std().item():.3f} min={f.min().item():+.2f} max={f.max().item():+.2f} nonfinite={nonfinite}")
        sys.stdout.flush()
    except Exception as _e:
        print(f"  🔬 {tag}: stats unavailable ({_e})")

try:
    import models.ltx2.ltx_pipelines.distilled as _distilled_probe_mod
    _orig_pipeline_upsample = _distilled_probe_mod.upsample_video
    _orig_pipeline_decode = _distilled_probe_mod.vae_decode_video_to_tensor

    def _probed_upsample_video(latent, video_encoder, upsampler):
        _latent_stats("Stage 1 latent", latent)
        in_dtype, in_device = latent.dtype, latent.device
        if AUX_ON_GPU1:
            # Upsampler + encoder statistics live on GPU 1: send the (small) latent over and back
            with torch.cuda.device(1):
                out = _orig_pipeline_upsample(latent=latent.to("cuda:1"), video_encoder=video_encoder, upsampler=upsampler)
        else:
            out = _orig_pipeline_upsample(latent=latent, video_encoder=video_encoder, upsampler=upsampler)
        out = out.to(device=in_device, dtype=in_dtype)  # upsample in FP32, hand back what the pipeline expects
        _latent_stats("Upscaled latent", out)
        return out

    from models.ltx2.ltx_core.model.video_vae.tiling import TilingConfig, SpatialTilingConfig, TemporalTilingConfig

    def _fast_vae_tiling(height, width):
        # Wan2GP decodes 120-frame chunks with a 45-frame overlap plus 512 px tiles with 128 px overlap:
        # a 10 s 832x480 clip decodes ~1.7x the pixels it keeps. During decode the transformer is
        # offloaded, so the FP32 decoder can use most of the card: whole frames when they fit, the
        # official LTX overlaps (24 frames / 64 px), and the longest chunks the VRAM budget allows.
        budget = torch.cuda.get_device_properties(0).total_memory - 1.6e9 - 3.0e9  # FP32 weights + margin
        bytes_per_pixel_frame = 230  # measured ~217 B on the T4 (12.6 GB peak at 832x480 x 128 frames)
        spatial = None
        tile_area = height * width
        if budget / (bytes_per_pixel_frame * tile_area) < 96:  # short chunks would waste more on overlap than tiling does
            spatial = SpatialTilingConfig(tile_size_in_pixels=512, tile_overlap_in_pixels=64)
            tile_area = min(height, 512) * min(width, 512)
        frames = int(budget / (bytes_per_pixel_frame * tile_area)) // 8 * 8
        frames = max(32, min(160, frames))
        return TilingConfig(spatial_config=spatial,
                            temporal_config=TemporalTilingConfig(tile_size_in_frames=frames, tile_overlap_in_frames=24))

    def _probed_decode(latent, video_decoder, tiling_config=None, *args, **kwargs):
        _latent_stats("Final latent -> VAE", latent)
        latent_tensor = latent[0] if isinstance(latent, list) else latent  # decode clears the list; keep it for a retry
        fast_config = _fast_vae_tiling(kwargs.get("expected_height") or 480, kwargs.get("expected_width") or 832)
        spatial_desc = "whole frame" if fast_config.spatial_config is None else "512 px tiles"
        print(f"  🎞️ VAE tiling: {spatial_desc}, {fast_config.temporal_config.tile_size_in_frames}-frame chunks (24 overlap)")
        # cudnn.benchmark stays OFF: autotuning every distinct FP32 conv3d shape of this decoder on a T4
        # takes many minutes and allocates huge trial workspaces.
        t_dec = time.time()
        torch.cuda.reset_peak_memory_stats(0)
        try:
            out = _orig_pipeline_decode([latent_tensor], video_decoder, fast_config, *args, **kwargs)
        except torch.OutOfMemoryError:
            print("  ⚠️ VAE out of memory with large chunks, retrying with Wan2GP's default tiling...")
            gc.collect()
            torch.cuda.empty_cache()
            out = _orig_pipeline_decode([latent_tensor], video_decoder, tiling_config, *args, **kwargs)
        print(f"  🎞️ VAE decode: {time.time() - t_dec:.1f}s (peak VRAM {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB)")
        return out

    _distilled_probe_mod.upsample_video = _probed_upsample_video
    _distilled_probe_mod.vae_decode_video_to_tensor = _probed_decode
except Exception as e:
    print(f"⚠️ Stage probe note: {e}")

# ==== Gemma 3 text encoder on the second T4 ====
# Gemma (13.2 GB) and the transformer (19.4 GB) do not both fit in 31 GB of RAM, so under mmgp they
# evict each other and get re-read from disk every generation (the ~70 s first step of each stage).
# Kaggle's "GPU T4 x2" gives us an idle second GPU: park Gemma there permanently instead.
TEXT_ENCODER_ON_GPU1 = False
if torch.cuda.device_count() > 1:
    try:
        print("📦 Moving Gemma 3 text encoder to GPU 1 (second T4)...")
        sys.stdout.flush()
        ltx2_model.text_encoder.to("cuda:1")
        # Weights alone nearly fill the 14.6 GB card, leaving no room for the forward pass.
        # The 1.9 GB token-embedding table (and lm_head, unused: LTX only reads hidden states)
        # go back to CPU: an embedding lookup of 1024 tokens there costs a few milliseconds.
        _gemma = ltx2_model.text_encoder.model
        _gemma_lm = _gemma.model if hasattr(_gemma, "model") else _gemma
        if getattr(_gemma, "lm_head", None) is not None:
            _gemma.lm_head = torch.nn.Identity()  # untied 1.9 GB copy, never used: drop it (frees RAM too)
        _embed = _gemma_lm.embed_tokens
        _embed.to("cpu")
        _orig_embed_forward = _embed.forward

        def _cpu_embed_forward(input_ids, *args, **kwargs):
            return _orig_embed_forward(input_ids.to("cpu"), *args, **kwargs).to("cuda:1")

        _embed.forward = _cpu_embed_forward
        # HF's model.device reports the first parameter (now the CPU embedding); encode_raw uses it
        # to place input_ids / attention_mask, so pin it to the GPU that runs the layers.
        _gemma.__class__ = type(_gemma.__class__.__name__, (_gemma.__class__,),
                                {"device": property(lambda self: torch.device("cuda", 1))})
        with torch.cuda.device(1):
            torch.cuda.empty_cache()
        pipe.pop("text_encoder", None)
        TEXT_ENCODER_ON_GPU1 = True
        _free1 = torch.cuda.mem_get_info(1)[0] / 1024**3
        print(f"✅ Gemma 3 resident on GPU 1 ({torch.cuda.memory_allocated(1) / 1024**3:.1f} GB weights, {_free1:.1f} GB free for encoding).")
    except Exception as e:
        print(f"⚠️ Gemma stays on GPU 0 with offloading ({e})")
        _gemma = ltx2_model.text_encoder.model
        if type(_gemma).__dict__.get("device") is not None and type(_gemma).__bases__:
            _gemma.__class__ = type(_gemma).__bases__[0]  # drop the pinned-device subclass
        _gemma_lm = _gemma.model if hasattr(_gemma, "model") else _gemma
        _gemma_lm.embed_tokens.__dict__.pop("forward", None)  # restore the stock embedding forward
        ltx2_model.text_encoder.to("cpu")
        with torch.cuda.device(1):
            torch.cuda.empty_cache()
    sys.stdout.flush()

# ==== Spatial upsampler + video encoder also live on GPU 1 ====
# mmgp keeps one model at a time on GPU 0: every time the upsampler (between the stages) or the
# video encoder (I2V start/end frames, once per stage) runs, the 9.9 GB transformer is evicted and
# reloaded (the ~75 s first step of Stage 2). Both fit next to Gemma on GPU 1, so they stay there.
AUX_ON_GPU1 = False
if TEXT_ENCODER_ON_GPU1 and pipe.get("spatial_upsampler") is not None:
    try:
        for _key in ("spatial_upsampler", "video_encoder"):
            pipe[_key].to("cuda:1")
        for _key in ("spatial_upsampler", "video_encoder"):
            pipe.pop(_key)
        AUX_ON_GPU1 = True
        print(f"✅ Spatial upsampler + video encoder resident on GPU 1 ({torch.cuda.mem_get_info(1)[0] / 1024**3:.1f} GB free).")
    except Exception as e:
        print(f"⚠️ Upsampler / encoder stay on GPU 0 with offloading ({e})")
        for _key in ("spatial_upsampler", "video_encoder"):
            getattr(ltx2_model, _key).to("cpu")
        with torch.cuda.device(1):
            torch.cuda.empty_cache()
    sys.stdout.flush()

# ==== Exact text encoding + text probe ====
# Gemma has extreme activation outlier channels. The Comfy Kitchen INT8 kernels quantize each
# activation row to int8, which flattens the normal channels to zero in such layers and degrades
# prompt understanding. Text encoding runs once per new prompt (it is cached), so it uses the exact
# int8-weight path; the video transformer keeps the fast kernels.
try:
    from optimum.quanto.tensor.weights import qbytes as _qbytes
except Exception:
    _qbytes = None
_orig_encode_text = _distilled_probe_mod.encode_text
_main_device = torch.device("cuda", 0)

def _exact_encode_text(text_encoder, prompts):
    saved_forward = None
    if _qbytes is not None and int8_backend is not None and getattr(int8_backend, "_original_forward", None) is not None:
        saved_forward = _qbytes.WeightQBytesLinearFunction.__dict__["forward"]
        _qbytes.WeightQBytesLinearFunction.forward = staticmethod(int8_backend._original_forward)
    try:
        with torch.cuda.device(1 if TEXT_ENCODER_ON_GPU1 else 0):
            out = _orig_encode_text(text_encoder, prompts)
    finally:
        if saved_forward is not None:
            _qbytes.WeightQBytesLinearFunction.forward = saved_forward
    out = [r._replace(hidden_states=tuple(h.to(_main_device) for h in r.hidden_states),
                      attention_mask=r.attention_mask.to(_main_device)) for r in out]
    for r in out:
        try:
            real = r.attention_mask[0].bool()
            h = r.hidden_states[-1][0][real].float()
            print(f"  🔬 Text embedding: {int(real.sum())} tokens | std={h.std().item():.2f} absmax={h.abs().max().item():.1f} nonfinite={int((~torch.isfinite(h)).sum())}")
        except Exception as _e:
            print(f"  🔬 Text embedding: stats unavailable ({_e})")
    sys.stdout.flush()
    return out

_distilled_probe_mod.encode_text = _exact_encode_text

# ==== Apply mmgp Profile 4 with Safe Asynchronous DMA Preloading ====
# Note: pinnedMemory=False avoids duplicating model weights in page-locked RAM,
# completely preventing Kaggle 31GB container RAM OOM (SIGKILL -9).
# asyncTransfers=True activates tune_preloading to keep ~10.8 GB transformer resident in VRAM
# (~55% of recurrent layers) and overlaps PCIe transfers with GPU execution, leaving ~4.0 GB free VRAM
# for peak Stage 2 832x480 attention activations without allocator contention.
print("⚡ Applying mmgp Profile 4 (BF16/int8 Transformer, FP32 VAE & Upscaler, safe async DMA)...")
sys.stdout.flush()

_budgets = {
    "transformer":       10800,  # leaves ~4 GB for 720p / long-clip activations
    "vae":               3500,   # FP32 decoder is 2x its BF16 size
    "spatial_upsampler": 2000,
    "video_encoder":     2000,
    "*":                 1500,
}
if not TEXT_ENCODER_ON_GPU1:
    _budgets["text_encoder"] = 3000
if AUX_ON_GPU1:
    _budgets.pop("spatial_upsampler")
    _budgets.pop("video_encoder")

offload.profile(
    pipe,
    profile_no=4,
    quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    pinnedMemory=False,
    asyncTransfers=True,
    verboseLevel=-1,
    budgets=_budgets,
)
offload.shared_state["_attention"] = "sdpa"

print("✅ Setup complete! Two-stage distilled pipeline active.\\n")
sys.stdout.flush()

# ==== Resolution and VAE Helpers ====
AUTO_ASPECT = "Auto (match input image)"

def get_resolution(base_res_str, aspect_ratio_str, ref_image=None):
    base_resolutions = {
        "1080p": 1088,
        "720p": 704,
        "540p": 544,
        "480p": 480,
    }
    ratios = {
        "16:9 Landscape": 16/9,
        "4:3 Standard": 4/3,
        "1:1 Square": 1.0,
        "3:4 Portrait": 3/4,
        "9:16 Portrait": 9/16,
    }
    base = base_resolutions.get(base_res_str, 704)
    if aspect_ratio_str == AUTO_ASPECT and ref_image is not None:
        # Follow the input image so it is not center-cropped when used as a start / end frame
        ratio = ref_image.width / ref_image.height
    else:
        ratio = ratios.get(aspect_ratio_str, 16/9)
    if ratio >= 1.0:
        height = base
        width = int(base * ratio)
    else:
        width = base
        height = int(base / ratio)
    width = (width // 32) * 32
    height = (height // 32) * 32
    return width, height

def get_vae_tile_size(height, width):
    ref_size = max(height, width)
    if ref_size <= 512:
        return 0, 1
    elif ref_size <= 960:
        return 512, 2
    else:
        return 384, 3

DEVICE = torch.device("cuda")

# ==== Video Generation Handler ====
@torch.inference_mode()
def Video_Generation(prompt, input_image_start, input_image_end, seed, duration_dropdown,
                     resolution_dropdown, aspect_ratio_dropdown,
                     generate_audio=True, progress=gr.Progress()):
    try:
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

        progress(0.02, desc="⚡ Initializing parameters...")

        duration_map = {
            "2 Seconds (49 frames)": 49,
            "3 Seconds (73 frames)": 73,
            "5 Seconds (121 frames)": 121,
            "10 Seconds (241 frames)": 241,
            "15 Seconds (361 frames)": 361,
            "20 Seconds (481 frames)": 481,
        }
        num_frames = duration_map.get(duration_dropdown, 121)
        frame_rate = 24.0

        if seed is None or seed < 0:
            seed = random.randint(0, 2**32 - 1)
        seed = int(seed)

        image_start = None
        image_end = None
        if input_image_start is not None:
            image_start = Image.open(input_image_start).convert("RGB")
        if input_image_end is not None:
            image_end = Image.open(input_image_end).convert("RGB")

        width, height = get_resolution(resolution_dropdown, aspect_ratio_dropdown,
                                       ref_image=image_start if image_start is not None else image_end)

        free_vram = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"\\n{'='*65}")
        print(f"🎬 Generating: {width}x{height} | {num_frames} frames ({num_frames/frame_rate:.1f}s) | Seed: {seed}")
        print(f"📝 Prompt: {prompt[:90]}...")
        if image_start is not None and image_end is not None:
            print("🖼️ Mode: Image-to-Video (First & Last Frame conditioning active)")
        elif image_start is not None:
            print("🖼️ Mode: Image-to-Video (First Frame conditioning active)")
        elif image_end is not None:
            print("🖼️ Mode: Image-to-Video (Last Frame conditioning active)")
        else:
            print("📝 Mode: Text-to-Video")
        print(f"⚡ Stage 1: {width//2}x{height//2} (8 steps) -> 2x Upscale -> Stage 2: {width}x{height} (3 steps)")
        print(f"💾 VRAM Free: {free_vram:.2f} GB")
        print(f"{'='*65}")
        sys.stdout.flush()

        vae_tile_size, _ = get_vae_tile_size(height, width)

        t_gen_start = time.time()
        timing_breakdown = {"s1": 0.0, "up": 0.0, "s2": 0.0, "vae": 0.0, "audio": 0.0}
        t_s1_start = [0.0]
        t_s2_start = [0.0]
        t_up_start = [0.0]
        t_vae_start = [0.0]

        total_steps = {1: 8, 2: 3}
        current_step = {1: 0, 2: 0}
        last_step_time = [time.time()]

        def cb(step, latent, is_start, override_num_inference_steps=None, pass_no=None, **kwargs):
            now = time.time()
            progress_title = kwargs.get("progress_title", "")
            progress_unit = kwargs.get("progress_unit", "")

            # 1. Gemma Text Encoding: update Gradio UI only, NO terminal stdout spam
            if progress_title == "Encoding Text Prompt" or progress_unit == "layers" or (pass_no not in (1, 2) and override_num_inference_steps == 48):
                step_cur = max(1, step + 1) if step is not None else 1
                step_tot = override_num_inference_steps or 48
                pct = min(1.0, step_cur / step_tot)
                progress(0.04 + 0.05 * pct, desc=f"⚡ Encoding Text Prompt: {step_cur}/{step_tot}")
                return

            # 2. VAE Tile Decoding: update Gradio UI only, NO terminal stdout spam
            if progress_unit == "tiles" or "VAE" in progress_title or "Decoding" in progress_title:
                tile_cur = max(1, step + 1) if step is not None else 1
                tile_tot = override_num_inference_steps or 6
                pct = min(1.0, tile_cur / tile_tot)
                progress(0.90 + 0.08 * pct, desc=f"🎞️ FP32 VAE Decoding: Tile {tile_cur}/{tile_tot}")
                return

            # 3. Filter out any other non-diffusion auxiliary callbacks
            if pass_no not in (1, 2):
                return

            # 4. Genuine Stage 1 & Stage 2 Diffusion
            p_no = pass_no
            if is_start:
                if override_num_inference_steps is not None and override_num_inference_steps > 0:
                    total_steps[p_no] = override_num_inference_steps
                current_step[p_no] = 0
                last_step_time[0] = now
                if p_no == 1:
                    print("  ✅ Text prompt encoded.")
                    t_s1_start[0] = now
                elif p_no == 2:
                    if t_up_start[0] > 0:
                        timing_breakdown["up"] = max(0.0, now - t_up_start[0])
                    t_s2_start[0] = now
                stg_label = "Stage 1 (Low-Res 8 steps)" if p_no == 1 else "Stage 2 (Refinement 3 steps)"
                print(f"\\n🚀 Starting {stg_label}...")
                sys.stdout.flush()
                return

            # Compute step index
            if step is not None and step >= 0:
                step_cur = step + 1
            else:
                current_step[p_no] += 1
                step_cur = current_step[p_no]
            current_step[p_no] = step_cur

            step_tot = max(total_steps.get(p_no, 8 if p_no == 1 else 3), 1)
            pct = min(1.0, step_cur / step_tot)
            elapsed_step = max(now - last_step_time[0], 0.01)
            last_step_time[0] = now
            speed_str = f"{elapsed_step:.1f}s/it" if elapsed_step >= 1.0 else f"{1.0/elapsed_step:.1f}it/s"
            eta_sec = max(0, int((step_tot - step_cur) * elapsed_step))
            eta_str = f"{eta_sec}s" if eta_sec < 60 else f"{eta_sec//60}m{eta_sec%60:02d}s"

            bar_len = 20
            filled = int(bar_len * pct)
            bar_str = "█" * filled + "░" * (bar_len - filled)
            vram_gb = torch.cuda.mem_get_info()[0] / 1024**3
            stg_short = "Stage 1" if p_no == 1 else "Stage 2"

            # Clean single-line progress update per step
            print(f"\\r  ⚡ [{stg_short}] |{bar_str}| {step_cur}/{step_tot} ({int(pct*100):3d}%) | {speed_str} | ETA: {eta_str} | Free VRAM: {vram_gb:.1f} GB", end="", flush=True)

            if step_cur >= step_tot:
                print()
                if p_no == 1:
                    timing_breakdown["s1"] = max(0.0, time.time() - t_s1_start[0])
                    t_up_start[0] = time.time()
                    print("  🔍 Running Spatial 2x Latent Upscaling...")
                elif p_no == 2:
                    timing_breakdown["s2"] = max(0.0, time.time() - t_s2_start[0])
                    t_vae_start[0] = time.time()
                    print("  ✅ Refinement complete.")
                sys.stdout.flush()

            # Gradio UI progress bar
            if p_no == 1:
                ui_pct = 0.10 + 0.55 * pct
                progress(ui_pct, desc=f"🎬 Stage 1 (Low-Res): Step {step_cur}/{step_tot} ({speed_str})")
            else:
                ui_pct = 0.70 + 0.18 * pct
                progress(ui_pct, desc=f"✨ Stage 2 (Refinement): Step {step_cur}/{step_tot} ({speed_str})")

        progress(0.04, desc="⚡ Encoding text prompt...")
        print("  ⚡ Encoding text prompt...")
        sys.stdout.flush()
        gen_kwargs = dict(
            input_prompt=prompt,
            image_start=image_start,
            height=height,
            width=width,
            frame_num=num_frames,
            fps=frame_rate,
            seed=seed,
            callback=cb,
            VAE_tile_size=vae_tile_size,
            guide_phases=2,
            # Same values Wan2GP uses for the distilled model (the generate() defaults are for the dev model)
            guide_scale=1.0,
            audio_cfg_scale=1.0,
            alt_guide_scale=1.0,
            input_video_strength=1.0,  # start / end frames are hard constraints
        )
        if image_end is not None:
            gen_kwargs["image_end"] = image_end

        result = ltx2_model.generate(**gen_kwargs)

        if result is None:
            return None, "❌ Generation failed or was interrupted."

        if t_vae_start[0] > 0:
            timing_breakdown["vae"] = max(0.0, time.time() - t_vae_start[0])

        progress(0.92, desc="🎞️ Saving video...")

        audio_data = None
        audio_sr = None

        if isinstance(result, dict):
            video_tensor = result.get("x")
            audio_data = result.get("audio")
            audio_sr = result.get("audio_sampling_rate", 24000)
        elif isinstance(result, tuple):
            video_tensor = result[0]
            if len(result) > 1:
                audio_data = result[1]
            if len(result) > 2:
                audio_sr = result[2]
        else:
            video_tensor = result

        if video_tensor is None or not torch.is_tensor(video_tensor):
            return None, f"❌ No video tensor produced. Received: {type(video_tensor)}"
        if not generate_audio:
            audio_data = None  # LTX-2 always co-generates audio; the checkbox controls whether it is kept

        video_tensor = video_tensor.cpu()
        gc.collect()
        torch.cuda.empty_cache()

        # Pixel probe: a healthy frame has std well above 20; the grey-collapse bug shows >70% of pixels in 120..136
        _px = video_tensor[:, ::max(1, video_tensor.shape[1] // 8)].float()
        _grey = ((_px >= 120) & (_px <= 136)).float().mean().item()
        print(f"  🔬 Decoded pixels: mean={_px.mean().item():.1f} std={_px.std().item():.1f} mid-grey fraction={_grey*100:.0f}%")
        del _px

        out_path = os.path.join(OUTPUT_DIR, f"ltx23_{time.strftime('%Y%m%d_%H%M%S')}_seed{seed}.mp4")
        if video_tensor.dtype == torch.uint8:
            # Native uint8 [0, 255] tensor from FP32 LTX VAE: stream directly without float re-quantization
            save_tensor = video_tensor.unsqueeze(0) if video_tensor.ndim == 4 else video_tensor
            save_video(
                tensor=save_tensor,
                save_file=out_path,
                fps=frame_rate,
                nrow=1,
            )
        else:
            save_video(
                tensor=video_tensor.unsqueeze(0).float() / 127.5 - 1.0,
                save_file=out_path,
                fps=frame_rate,
                nrow=1,
                normalize=True,
                value_range=(-1, 1),
            )

        if audio_data is not None:
            t_aud_start = time.time()
            progress(0.96, desc="🎵 Muxing synchronized audio track...")
            try:
                import soundfile as sf
                audio_tmp = tempfile.mktemp(suffix=".wav")
                if isinstance(audio_data, np.ndarray):
                    audio_np = audio_data
                    if audio_np.ndim == 2 and audio_np.shape[0] <= 2:
                        audio_np = audio_np.T
                    sr = int(audio_sr) if audio_sr else 24000
                    sf.write(audio_tmp, audio_np, sr)
                elif torch.is_tensor(audio_data):
                    import torchaudio
                    audio_cpu = audio_data.cpu().float()
                    if audio_cpu.dim() == 1:
                        audio_cpu = audio_cpu.unsqueeze(0)
                    if audio_cpu.dim() == 3:
                        audio_cpu = audio_cpu.squeeze(0)
                    sr = int(audio_sr) if audio_sr else 24000
                    torchaudio.save(audio_tmp, audio_cpu, sr)

                final_path = out_path.replace(".mp4", "_with_audio.mp4")
                subprocess.run([
                    "ffmpeg", "-y", "-i", out_path, "-i", audio_tmp,
                    "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                    "-shortest", final_path
                ], check=True, capture_output=True)
                if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
                    os.remove(out_path)  # keep only the version with audio in the outputs folder
                    out_path = final_path
                if os.path.exists(audio_tmp):
                    os.remove(audio_tmp)
                timing_breakdown["audio"] = max(0.0, time.time() - t_aud_start)
            except Exception as e:
                print(f"  ⚠️ Audio mux skipped: {e}")

        total_time = time.time() - t_gen_start
        total_m = int(total_time // 60)
        total_s = int(total_time % 60)
        time_str = f"{total_m}m {total_s:02d}s ({total_time:.1f}s)" if total_m > 0 else f"{total_time:.1f}s"

        s1_s = timing_breakdown.get("s1", 0.0)
        up_s = timing_breakdown.get("up", 0.0)
        s2_s = timing_breakdown.get("s2", 0.0)
        vae_s = timing_breakdown.get("vae", 0.0)
        aud_s = timing_breakdown.get("audio", 0.0)

        s1_spd = f"{s1_s/8:.1f}s/it" if s1_s > 0 else "N/A"
        s2_spd = f"{s2_s/3:.1f}s/it" if s2_s > 0 else "N/A"

        audio_status = "🎵 Synchronized 48kHz Stereo AAC Included" if audio_data is not None else "🔇 Audio Disabled / None"
        if image_start is not None and image_end is not None:
            mode_desc = "🖼️ Image-to-Video (Start & End Frame Conditioning)"
        elif image_start is not None:
            mode_desc = "🖼️ Image-to-Video (Start Frame Conditioning)"
        elif image_end is not None:
            mode_desc = "🖼️ Image-to-Video (End Frame Conditioning)"
        else:
            mode_desc = "📝 Text-to-Video"

        try:
            peak_vram_gb = torch.cuda.max_memory_allocated() / (1024**3)
            hw_str = f"GPU T4 x2 (Peak VRAM: {peak_vram_gb:.1f} GB / 14.6 GB)"
        except Exception:
            hw_str = "GPU T4 x2 (14.6 GB)"

        perf_line = f"Stage 1 (8 steps): {s1_s:.1f}s ({s1_spd}) | 2x Upscale: {up_s:.1f}s | Stage 2 (3 steps): {s2_s:.1f}s ({s2_spd}) | FP32 VAE: {vae_s:.1f}s"
        if aud_s > 0:
            perf_line += f" | Audio Mux: {aud_s:.1f}s"

        status_lines = [
            f"✅ Generation Complete in {time_str}!",
            "─────────────────────────────────────────────────────────────────",
            f"⚡ Performance Breakdown: {perf_line}",
            f"📐 Resolution & Frames: {width}x{height} ({aspect_ratio_dropdown}) | {num_frames} frames ({num_frames/frame_rate:.1f}s @ {int(frame_rate)}fps)",
            f"🎲 Seed: {seed} | Mode: {mode_desc}",
            f"🎵 Audio: {audio_status}",
            f"💾 Hardware: {hw_str}",
            f"📁 Output Video: {os.path.basename(out_path)}"
        ]
        status_text = "\\n".join(status_lines)

        del video_tensor
        gc.collect()
        torch.cuda.empty_cache()

        progress(1.0, desc="✅ Generation complete!")
        print(f"\\n{'='*65}")
        print(f"🎬 Video Generation Completed Successfully!")
        print(f"⏱️ Total Time: {time_str} | Resolution: {width}x{height} | Seed: {seed}")
        print(f"⚡ {perf_line}")
        print(f"📁 Output Video: {out_path}")
        print(f"{'='*65}\\n")
        sys.stdout.flush()
        return out_path, status_text

    except Exception as e:
        traceback.print_exc()
        gc.collect()
        torch.cuda.empty_cache()
        return None, f"❌ Error: {str(e)}"

# ==== Gradio UI (AIQUEST Academy Branded) ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.brand-footer { text-align: center; color: #6b7280; font-size: 12px; margin-top: 24px; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎬 LTX-2.3 22B Distilled Video Generator</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Kaggle GPU T4 x2 Edition</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""

_int8_label = {"kitchen": "Comfy Kitchen INT8 kernels", "triton": "Triton INT8 kernels"}.get(
    getattr(int8_backend, "_backend", "pytorch") if int8_backend is not None else "pytorch", "PyTorch INT8 (no fast kernels)")
_gpu_layout = "Gemma + upsampler on GPU 1" if AUX_ON_GPU1 else ("Gemma on GPU 1" if TEXT_ENCODER_ON_GPU1 else "single GPU")
STATUS_HTML = f'<div class="status-line">⚡ {_int8_label} &nbsp;|&nbsp; {_gpu_layout} &nbsp;|&nbsp; FP32 VAE &nbsp;|&nbsp; 8 + 3 step distilled pipeline</div>'

FOOTER_HTML = """
<div class="brand-footer">⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved</div>
"""

with gr.Blocks(theme=gr.themes.Soft(), css=CSS, title="LTX-2.3 22B Video Generator | AIQUEST Academy") as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)

    gr.Markdown(
        "💡 **Tips:** write detailed prompts (subject, action, setting, camera, lighting). "
        "480p / 540p handle 5-10 s clips; 720p works best at 3-5 s."
    )

    with gr.Column():
        prompt = gr.Textbox(label="🎬 Prompt", lines=3,
                   placeholder="A cinematic shot of a red fox walking through a snowy forest, soft ambient light...")

        with gr.Accordion("🖼️ Image-to-Video Conditioning (Optional)", open=False):
            with gr.Row():
                input_image_start = gr.Image(type="filepath", label="Start Frame (optional)")
                input_image_end = gr.Image(type="filepath", label="End Frame (optional)")
            gr.Markdown("*Start frame only = image-to-video. Start + end = first/last-frame interpolation. End only = the video lands on that frame. Describe the motion in the prompt.*")

        with gr.Row():
            seed = gr.Number(label="🎲 Seed (-1 for Random)", value=-1, precision=0)
            duration_dropdown = gr.Dropdown(
                label="⏱️ Duration",
                choices=[
                   "2 Seconds (49 frames)",
                   "3 Seconds (73 frames)",
                   "5 Seconds (121 frames)",
                   "10 Seconds (241 frames)",
                   "15 Seconds (361 frames)",
                   "20 Seconds (481 frames)",
                ],
                value="5 Seconds (121 frames)",
            )
            generate_audio_cb = gr.Checkbox(label="🎵 Generate Synchronized Audio", value=True)

        with gr.Row():
            resolution_dropdown = gr.Dropdown(
                label="📐 Base Quality",
                choices=["1080p", "720p", "540p", "480p"],
                value="480p",
            )
            aspect_ratio_dropdown = gr.Dropdown(
                label="📏 Aspect Ratio",
                choices=[AUTO_ASPECT, "16:9 Landscape", "4:3 Standard", "1:1 Square", "3:4 Portrait", "9:16 Portrait"],
                value=AUTO_ASPECT,
                info="Auto follows the start/end image (16:9 for text-to-video)",
            )

        with gr.Row():
            gen_btn = gr.Button("🎬 Generate Video", variant="primary", size="lg", elem_id="gen-btn")
            stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
            clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

        video_out = gr.Video(label="🎥 Generated Video")
        latest_btn = gr.Button("📂 Load Latest Video (use if the UI showed an error)", variant="secondary")
        status_out = gr.Textbox(label="ℹ️ Generation Telemetry & Status", lines=7, max_lines=12, interactive=False)

        def load_latest_video():
            videos = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.mp4")), key=os.path.getmtime)
            if not videos:
                return None, f"No videos in {OUTPUT_DIR} yet."
            return videos[-1], f"📁 Loaded latest video: {videos[-1]}"

        latest_btn.click(fn=load_latest_video, outputs=[video_out, status_out])

        gen_event = gen_btn.click(
            fn=Video_Generation,
            inputs=[prompt, input_image_start, input_image_end, seed, duration_dropdown,
                    resolution_dropdown, aspect_ratio_dropdown, generate_audio_cb],
            outputs=[video_out, status_out],
        )
        stop_btn.click(fn=None, cancels=[gen_event])
        clear_btn.click(
            fn=lambda: (None, "Ready for generation.", "", -1, None, None, True),
            outputs=[video_out, status_out, prompt, seed, input_image_start, input_image_end, generate_audio_cb],
        )

    gr.HTML(FOOTER_HTML)

# ==== Cloudflare quick tunnel: a second public link in case the Gradio share link fails ====
SERVER_PORT = 7860

def start_cloudflare_tunnel(port):
    import re
    import stat
    import threading
    import urllib.request
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return

    def _watch():
        for line in tunnel.stdout:
            match = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", line)
            if match:
                print(f"* Cloudflare tunnel URL: {match.group(0)}  (use this if the Gradio link does not load)")
                sys.stdout.flush()
                break
        for _ in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            pass

    threading.Thread(target=_watch, daemon=True).start()

print("\\n🚀 Launching Gradio Web Interface...")
sys.stdout.flush()
start_cloudflare_tunnel(SERVER_PORT)
demo.queue()
demo.launch(
    server_name="127.0.0.1",
    server_port=SERVER_PORT,
    share=True,
    inline=False,
    debug=False,
    show_error=True,
    max_threads=1,
    ssr_mode=False,
    allowed_paths=[OUTPUT_DIR],
)
'''

with open("run_ltx.py", "w", encoding="utf-8") as f:
    f.write(RUN_SCRIPT_CODE)
print("✅ run_ltx.py generated successfully.")

# Clean memory and caches before launch
gc.collect()
os.system("echo 3 | sudo -n tee /proc/sys/vm/drop_caches > /dev/null 2>&1")

# Launch generation engine with real-time log streaming & C++ error suppression
env = dict(os.environ,
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True,garbage_collection_threshold:0.6",
           TOKENIZERS_PARALLELISM="false",
           TF_CPP_MIN_LOG_LEVEL="3",
           TF_ENABLE_ONEDNN_OPTS="0",
           TQDM_DISABLE="1",
           PYTHONWARNINGS="ignore")

IGNORE_PATTERNS = [
    "absl::InitializeLog",
    "Unable to register cuDNN factory",
    "Unable to register cuBLAS factory",
    "Unable to register cuFFT factory",
    "computation placer already registered",
    "MessageFactory",
    "GetPrototype",
    "Attempting to register factory",
]

print("🚀 Starting LTX-2.3 22B Distilled engine...\n")
proc = subprocess.Popen([sys.executable, "-u", "run_ltx.py"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=env)
for line in iter(proc.stdout.readline, ""):
    if any(pattern in line for pattern in IGNORE_PATTERNS):
        continue
    print(line, end="", flush=True)
proc.wait()

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---